# SMS Spam Detection

End-to-end pipeline: data loading -> EDA -> text preprocessing -> feature engineering
-> model training -> threshold tuning -> error analysis -> try it yourself.

All heavy lifting lives in `src/`; this notebook imports from there and displays results.

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# Make src/ importable when running from notebooks/
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

from src.config import PROJECT_ROOT, FIGURES_DIR

RESULTS = json.loads((PROJECT_ROOT / 'reports' / 'results.json').read_text())
print('Loaded results.json')
print(f"Dataset: {RESULTS['dataset']['total_rows']:,} rows after dedup")

Loaded results.json
Dataset: 5,169 rows after dedup


## 1. Load data & EDA

In [2]:
from src.data import load_raw, deduplicate, split, spam_rate

df_raw = load_raw()
print(f'Raw rows:  {len(df_raw):,}')
print(f'Spam rate: {df_raw["target"].mean()*100:.2f}%')
print(f'Columns:   {list(df_raw.columns)}')
df_raw.head(5)

Raw rows:  5,572
Spam rate: 13.41%
Columns:   ['label', 'text', 'target']


,label,text,target
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


### 1.1 Class balance

In [3]:
fig, ax = plt.subplots(figsize=(8, 4))
img = mpimg.imread(str(FIGURES_DIR / 'class_balance.png'))
ax.imshow(img); ax.axis('off')
plt.tight_layout()
plt.show()

### 1.2 Message length by class

In [4]:
df_raw['length'] = df_raw['text'].str.len()
print('Median message length:')
print(df_raw.groupby('label')['length'].median())

fig, ax = plt.subplots(figsize=(10, 4))
img = mpimg.imread(str(FIGURES_DIR / 'length_distribution.png'))
ax.imshow(img); ax.axis('off')
plt.tight_layout()
plt.show()

Median message length:
label
ham      52.0
spam    149.0
Name: length, dtype: float64


### 1.3 Duplicate messages

In [5]:
n_dupes  = RESULTS['dataset']['total_rows']
n_orig   = len(df_raw)
removed  = n_orig - n_dupes
print(f'Original rows:        {n_orig:,}')
print(f'Duplicates removed:   {removed}')
print(f'Unique rows:          {n_dupes:,}')
print()
print('Most-repeated texts (top 5):')
dupe_counts = df_raw.groupby('text').size().sort_values(ascending=False)
dupe_counts[dupe_counts > 1].head(5)

Original rows:        5,572
Duplicates removed:   403
Unique rows:          5,169

Most-repeated texts (top 5):


text
Sorry, I'll call later                                                                                                                                                                 30
I cant pick the phone right now. Pls send a message                                                                                                                                    12
Ok...                                                                                                                                                                                  10
Your opinion about me? 1. Over 2. Jada 3. Kusruthi 4. Lovable 5. Silent 6. Spl character 7. Not matured 8. Stylish 9. Simple Pls reply..                                                4
Wen ur lovable bcums angry wid u, dnt take it seriously.. Coz being angry is d most childish n true way of showing deep affection, care n luv!.. kettoda manda... Have nice day da.     4
dtype: int64

Duplicates are removed **before** splitting so the same message cannot appear
in both train and test sets (no leakage).

## 2. Text preprocessing comparison

In [6]:
from src.preprocess import clean_minimal, clean_heavy

examples = [
    'WINNER!! As a valued network customer you have been selected to receive a 900 prize reward!',
    'Hey, are you coming to the meeting at 3pm? Let me know!',
    'FREE entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005.',
]
print('Minimal cleaning (lowercase only):')
for t in examples:
    print(f'  -> {clean_minimal(t)}')

print()
print('Heavy cleaning (digit normalise + punct strip + stopwords + stemming):')
for t in examples:
    print(f'  -> {clean_heavy(t)}')

Minimal cleaning (lowercase only):
  -> winner!! as a valued network customer you have been selected to receive a 900 prize reward!
  -> hey, are you coming to the meeting at 3pm? let me know!
  -> free entry in 2 a wkly comp to win fa cup final tkts 21st may 2005.

Heavy cleaning (digit normalise + punct strip + stopwords + stemming):
  -> winner!! valu network custom select receiv num prize reward!
  -> hey come meet num pm? let know!
  -> free entri num wkli comp win fa cup final tkt num st may num


**Grid search result:** minimal cleaning won (CV PR-AUC 0.9714).

Why heavier cleaning does not help here:
- TF-IDF with n-grams already down-weights common words (like a soft stopword removal).
- Stemming can merge distinct spam tokens: 'won' and 'win' share a root but carry different signal strengths in context.
- Removing punctuation strips exclamation marks that are strong spam indicators.

## 3. The accuracy paradox

In [7]:
dummy_acc = RESULTS['dummy']['accuracy']
print(f'DummyClassifier (always predicts ham):')
print(f'  Accuracy = {dummy_acc*100:.2f}%  |  Spam caught = 0%')
print()
print('A model that never raises an alarm scores 87.33% accuracy.')
print('Accuracy is useless as the primary metric here.')

DummyClassifier (always predicts ham):
  Accuracy = 87.33%  |  Spam caught = 0%

A model that never raises an alarm scores 87.33% accuracy.
Accuracy is useless as the primary metric here.


## 4. Model training & comparison

In [8]:
nb  = RESULTS['naive_bayes']
lr  = RESULTS['logistic_regression']
svm = RESULTS['linear_svm']

rows = [
    {'Model': 'DummyClassifier (always ham)',
     'CV PR-AUC': '--',
     'Test PR-AUC': '--',
     'Precision': '--',
     'Recall': f"{RESULTS['dummy']['recall']:.4f}",
     'F1': '--'},
    {'Model': 'Logistic Regression',
     'CV PR-AUC': f"{lr['cv_pr_auc_mean']:.4f} +/- {lr['cv_pr_auc_std']:.4f}",
     'Test PR-AUC': f"{lr['test_pr_auc']:.4f}",
     'Precision': f"{lr['test_precision']:.4f}",
     'Recall':    f"{lr['test_recall']:.4f}",
     'F1':        f"{lr['test_f1']:.4f}"},
    {'Model': 'Linear SVM (calibrated)',
     'CV PR-AUC': f"{svm['cv_pr_auc_mean']:.4f} +/- {svm['cv_pr_auc_std']:.4f}",
     'Test PR-AUC': f"{svm['test_pr_auc']:.4f}",
     'Precision': f"{svm['test_precision']:.4f}",
     'Recall':    f"{svm['test_recall']:.4f}",
     'F1':        f"{svm['test_f1']:.4f}"},
    {'Model': 'Naive Bayes (tuned)',
     'CV PR-AUC': f"{nb['cv_pr_auc_mean']:.4f} +/- {nb['cv_pr_auc_std']:.4f}",
     'Test PR-AUC': f"{nb['test_pr_auc']:.4f}",
     'Precision': f"{nb['test_precision']:.4f}",
     'Recall':    f"{nb['test_recall']:.4f}",
     'F1':        f"{nb['test_f1']:.4f}"},
]
pd.DataFrame(rows).set_index('Model')

,CV PR-AUC,Test PR-AUC,Precision,Recall,F1
Model,,,,,
DummyClassifier (always ham),--,--,--,0.0000,--
Logistic Regression,0.9739 +/- 0.0040,0.9636,0.9516,0.9008,0.9255
Linear SVM (calibrated),0.9780 +/- 0.0046,0.9710,0.9520,0.9084,0.9297
Naive Bayes (tuned),0.9714 +/- 0.0113,0.9755,0.9528,0.9237,0.9380


In [9]:
print('Best NB params from grid search:')
for k, v in nb['best_params'].items():
    print(f'  {k}: {v}')

Best NB params from grid search:
  cleaning: minimal
  ngram_range: [1, 2]
  max_features: 10000
  alpha: 0.1


### 4.1 PR curves

In [10]:
fig, ax = plt.subplots(figsize=(8, 5))
img = mpimg.imread(str(FIGURES_DIR / 'pr_curves.png'))
ax.imshow(img); ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Threshold tuning (NB, validation set)

For spam detection the **cost asymmetry** runs the opposite way from fraud detection:
- **False positive** (real message sent to spam): user loses a message they wanted. Usually **worse**.
- **False negative** (spam gets through): minor annoyance.

The business priority determines which threshold to use.

In [11]:
fig, ax = plt.subplots(figsize=(10, 4))
img = mpimg.imread(str(FIGURES_DIR / 'precision_recall_vs_threshold.png'))
ax.imshow(img); ax.axis('off')
plt.tight_layout()
plt.show()

### 5.1 Operating points

In [12]:
ops = RESULTS['operating_points']
rows = []
for key, op in ops.items():
    rows.append({
        'Operating point': op['name'],
        'Threshold': f"{op['threshold']:.3f}",
        'Precision': f"{op['precision']:.4f}",
        'Recall':    f"{op['recall']:.4f}",
        'F1':        f"{op['f1']:.4f}",
    })
pd.DataFrame(rows).set_index('Operating point')

,Threshold,Precision,Recall,F1
Operating point,,,,
Max F1,0.224,0.9661,0.8702,0.9157
Precision ≥ 99%,0.559,0.9907,0.8168,0.8954
Recall ≥ 95%,0.040,0.7143,0.9542,0.8170


In [13]:
fig, ax = plt.subplots(figsize=(8, 5))
img = mpimg.imread(str(FIGURES_DIR / 'pr_curve_operating_points.png'))
ax.imshow(img); ax.axis('off')
plt.tight_layout()
plt.show()

## 6. Final test-set evaluation

In [14]:
print(f"Chosen threshold (max F1 on validation): {nb['chosen_threshold']:.4f}")
print(f"  Test PR-AUC  : {nb['test_pr_auc']:.4f}")
print(f"  ROC-AUC      : {nb['test_roc_auc']:.4f}")
print(f"  Precision    : {nb['test_precision']:.4f}")
print(f"  Recall       : {nb['test_recall']:.4f}")
print(f"  F1           : {nb['test_f1']:.4f}")
print(f"  TP={nb['test_tp']}  FP={nb['test_fp']}  TN={nb['test_tn']}  FN={nb['test_fn']}")

Chosen threshold (max F1 on validation): 0.2243
  Test PR-AUC  : 0.9755
  ROC-AUC      : 0.9923
  Precision    : 0.9528
  Recall       : 0.9237
  F1           : 0.9380
  TP=121  FP=6  TN=897  FN=10


In [15]:
fig, ax = plt.subplots(figsize=(5, 4))
img = mpimg.imread(str(FIGURES_DIR / 'confusion_matrix.png'))
ax.imshow(img); ax.axis('off')
plt.tight_layout()
plt.show()

## 7. Error analysis

In [16]:
print('FALSE POSITIVES -- ham messages wrongly flagged as spam:')
for ex in RESULTS['false_positives']:
    print(f"  [p={ex['prob']:.3f}] {ex['text'][:100]}")

print()
print('FALSE NEGATIVES -- spam messages that slipped through:')
for ex in RESULTS['false_negatives']:
    print(f"  [p={ex['prob']:.3f}] {ex['text'][:100]}")

FALSE POSITIVES -- ham messages wrongly flagged as spam:
  [p=0.288] K k:) sms chat with me.
  [p=0.291] Have you heard from this week?
  [p=0.366] Waiting for your call.
  [p=0.279] Hey...Great deal...Farm tour 9am to 5pm $95/pax, $50 deposit by 16 May
  [p=0.604] Received, understood n acted upon!

FALSE NEGATIVES -- spam messages that slipped through:
  [p=0.034] Hi if ur lookin 4 saucy daytime fun wiv busty married woman Am free all next week Chat now 2 sort ti
  [p=0.155] ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE MINS. INDIA CUST SERVs SED YES. L8ER GOT MEGA BILL. 3 
  [p=0.017] Hi its LUCY Hubby at meetins all day Fri & I will B alone at hotel U fancy cumin over? Pls leave msg
  [p=0.049] Missed call alert. These numbers called but left no message. 07008009200
  [p=0.003] Babe: U want me dont u baby! Im nasty and have a thing 4 filthyguys. Fancy a rude time with a sexy b


**Why these are hard:**

- *False positives*: short messages with rare trigger words ('sms chat', 'call', 'deal'), or messages containing a price like $95. The NB model has seen these tokens mostly in spam context.
- *False negatives*: sexually explicit spam avoids the classic keyword pattern (no FREE, WIN, PRIZE) and reads more like personal messages. The lowest-probability cases use highly informal SMS dialect.

## 8. Interpretation -- top discriminative words

In [17]:
fig, ax = plt.subplots(figsize=(12, 5))
img = mpimg.imread(str(FIGURES_DIR / 'top_words.png'))
ax.imshow(img); ax.axis('off')
plt.tight_layout()
plt.show()

In [18]:
print('Top 10 spam indicators (NB log-probability ratio):')
for word, score in RESULTS['top_spam_words'][:10]:
    print(f'  {word:<20} {score:.3f}')

print()
print('Top 10 ham indicators:')
for word, score in RESULTS['top_ham_words'][:10]:
    print(f'  {word:<20} {score:.3f}')

Top 10 spam indicators (NB log-probability ratio):
  claim                5.683
  prize                5.613
  have won             5.138
  your mobile          5.092
  18                   5.089
  150p                 5.061
  to claim             5.012
  500                  5.009
  guaranteed           4.988
  co                   4.978

Top 10 ham indicators:
  lt                   4.155
  gt                   4.141
  lor                  4.119
  lt gt                4.035
  da                   4.023
  he                   4.016
  she                  3.689
  i_                   3.665
  later                3.631
  too                  3.586


**Observations:**

Spam tokens: 'claim', 'prize', 'have won', 'your mobile', 'nokia' -- classic 2000s UK/Singapore lottery and ringtone spam vocabulary.

Ham tokens: 'lt' and 'gt' are XML escape artefacts from the original dataset encoding; 'lor', 'da', 'lol' are Singaporean/informal English markers.

## 9. Try it yourself

Edit the `message` variable below and re-run the cell.

In [19]:
from src.predict import predict_message

# Edit this message
message = 'Congratulations! You have been selected to claim a FREE 500 prize. Call 08001234567 NOW!'

result = predict_message(message)
print(f'Message  : {message}')
print(f'Label    : {result["label"].upper()}')
print(f'Spam prob: {result["probability"]:.4f}  (threshold = {result["threshold"]:.4f})')

Message  : Congratulations! You have been selected to claim a FREE 500 prize. Call 08001234567 NOW!
Label    : SPAM
Spam prob: 1.0000  (threshold = 0.2243)


In [20]:
# Ham example
message2 = 'Hey, are you free for lunch tomorrow around 1pm?'
result2  = predict_message(message2)
print(f'Message  : {message2}')
print(f'Label    : {result2["label"].upper()}')
print(f'Spam prob: {result2["probability"]:.4f}  (threshold = {result2["threshold"]:.4f})')

Message  : Hey, are you free for lunch tomorrow around 1pm?
Label    : HAM
Spam prob: 0.0334  (threshold = 0.2243)
